In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
from yellowbrick.cluster import KElbowVisualizer
import matplotlib.pyplot as plt
import pandas as pd 
import seaborn as sns
from sklearn.cluster import KMeans
from sksurv.base import SurvivalAnalysisMixin as s
from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score
from sksurv.preprocessing import encode_categorical
from sksurv.datasets import load_gbsg2
from sksurv.functions import StepFunction
from sksurv.linear_model import CoxPHSurvivalAnalysis, CoxnetSurvivalAnalysis
from sksurv.ensemble import (ComponentwiseGradientBoostingSurvivalAnalysis, 
                            RandomSurvivalForest, 
                            ExtraSurvivalTrees, 
                            GradientBoostingSurvivalAnalysis, 
                            ExtraSurvivalTrees)
from sksurv.meta import EnsembleSelection, EnsembleSelectionRegressor
from sksurv.metrics import integrated_brier_score
from matplotlib.colors import ListedColormap
from mlxtend.evaluate import paired_ttest_5x2cv
from mlxtend.evaluate import combined_ftest_5x2cv
from lifelines import KaplanMeierFitter
from scipy.cluster import hierarchy
from lifelines.statistics import logrank_test, multivariate_logrank_test, pairwise_logrank_test
from sklearn import preprocessing
from sklearn.model_selection import StratifiedKFold, KFold
from lifelines.plotting import add_at_risk_counts
import scipy.stats
import sklearn
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import optuna
from sklearn.model_selection import cross_val_score
from sksurv.metrics import integrated_brier_score
from lifelines import CoxPHFitter
from lifelines.statistics import proportional_hazard_test
import scipy.stats as stats
from statsmodels.stats.outliers_influence import variance_inflation_factor 
from sklearn.preprocessing import StandardScaler
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'

In [2]:
# OUS: Train data
OUS_D1 = pd.read_csv('OUS_D1.csv')
OUS_D2 = pd.read_csv('OUS_D2.csv')
OUS_D3 = pd.read_csv('OUS_D3.csv')
OUS_DFS_target = pd.read_csv('OUS_DFS_target.csv')
OUS_OS_target = pd.read_csv('OUS_OS_target.csv')
response_OUS = pd.read_csv('response_ous.csv', sep=';')

# MAASTRO: Test data 
MAASTRO_D1 = pd.read_csv('MAASTRO_D1.csv')
MAASTRO_D2 = pd.read_csv('MAASTRO_D2.csv')
MAASTRO_D3 = pd.read_csv('MAASTRO_D3.csv')
MAASTRO_DFS_target = pd.read_csv('MAASTRO_DFS_target.csv')
MAASTRO_OS_target = pd.read_csv('MAASTRO_OS_target.csv')
response_MAASTRO = pd.read_csv('maastro_response_full.csv', sep=',')

In [3]:
# Need to choose patient_id from OUS_D1 in response_OUS
data = list(OUS_D1['patient_id'])
mask = response_OUS['patient_id'].isin(data)
response_OUS = response_OUS[mask] 

# Merge OUS_D1 with response_OUS
clinical_train = pd.merge(OUS_D1, response_OUS, on='patient_id', how='inner')
clinical_train = clinical_train.loc[:, ~clinical_train.columns.isin(['OS', 'event_OS', 'LRC', 'event_LRC'])]

In [4]:
# Drop patient_id column
clinical_train = clinical_train.drop('patient_id', axis=1)

In [5]:
# Check null values in D1
clinical_train.isnull().sum().sum()

0

## Test dataset: MAASTRO 

In [6]:
(MAASTRO_D1['patient_id'] == MAASTRO_OS_target['patient_id']).sum()

99

In [7]:
# Rename the column name of response_MAASTRO 
response_MAASTRO.rename(columns = {'Index' : 'patient_id'}, inplace = True)

In [8]:
# need to choose patient_id from MAASTRO_D1 in response_MAASTRO
data = list(MAASTRO_D1['patient_id'])
mask = response_MAASTRO['patient_id'].isin(data)
response_MAASTRO = response_MAASTRO[mask] 

In [9]:
# Merge MAASTRO_D1 with response_MAASTRO
clinical_test = pd.merge(MAASTRO_D1, response_MAASTRO, on='patient_id', how='inner')
clinical_test = clinical_test.loc[:, ~clinical_test.columns.isin(['OS', 'OS_event', 'LRC', 'LRC_event'])]

In [10]:
# Drop patient_id column
clinical_test = clinical_test.drop('patient_id', axis=1)

In [11]:
# Check if some rows have null values in OS, OS_event -> Remove those rows
clinical_test[clinical_test.isnull().any(axis=1)]
clinical_test = clinical_test.dropna(how='any',axis=0) 

,age,female,cavum_oris,oropharynx,hypopharynx,larynx,histgrade_high,hpv_related,charlson,pack_years,uicc8_III-IV,SUVpeak,MTV,TLG,DFS,DFS_event


In [12]:
# X
X = clinical_train.loc[:, ~clinical_train.columns.isin(['DFS', 'event_DFS'])]

# y 
y = clinical_train.loc[:, ['DFS', 'event_DFS']]

In [13]:
# Set lower, upper time point and times for IBS calculation later 

# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

# Shape
print('X_train: ', X.shape)
print('y_train: ', y.shape)

clinical_test.rename(columns = {'DFS_event' : 'event_DFS'}, inplace = True)

# X
X_MAASTRO = clinical_test.loc[:, ~clinical_test.columns.isin(['DFS', 'event_DFS'])]

# y y_MAASTRO
y_MAASTRO = clinical_test.loc[:, ['DFS', 'event_DFS']]
lower, upper = np.percentile(y_MAASTRO['DFS'], [10, 90])
times = np.arange(lower, upper)

# y into array 
lists = [] 
for i, j in zip(y_MAASTRO['event_DFS'], y_MAASTRO['DFS']): 
    lists.append((i, j))

y_MAASTRO = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

clinical_test.shape

X_train:  (139, 14)
y_train:  (139,)


(99, 16)

# Feature Selection: PLSR

In [14]:
selected_features = [ "hpv_related" ,           
"oropharynx",              
"cavum_oris"]

# Selecting features in the DataFrame
X_rent = X[selected_features]

In [15]:
# Selecting features in the DataFrame
X_rent = X[selected_features]
X_new = X_rent.copy()

X_MAASTRO_rent = X_MAASTRO[selected_features]
MAASTRO_new = X_MAASTRO_rent.copy()

# Standardization

In [16]:
# Standardize X_new, the new data with the selected features only 
categorical_columns = ['female', 
                        'cavum_oris',
                        'oropharynx',
                        'hypopharynx',
                        'larynx',
                        'histgrade_high',
                        'hpv_related',
                        'charlson',
                        'uicc8_III-IV']

# Set the columns_to_drop which are categorical 
columns_to_drop = [col for col in X_new.columns if col in categorical_columns]

# Drop the columns if they exist in the DataFrame and save the numeric part in X_new_numeric
X_new_categoric = X_new[columns_to_drop]
X_new_numeric = X_new.drop(columns=columns_to_drop, inplace=False)
""" 
# Do the standardization for the numeric part 
scaler = StandardScaler() 
X_new_numeric_columns = X_new_numeric.columns
X_new_numeric_index = X_new_numeric.index 
X_new_numeric_std = scaler.fit_transform(X_new_numeric)
X_new_numeric_std = pd.DataFrame(X_new_numeric_std,
                                 columns=X_new_numeric_columns, 
                                 index=X_new_numeric_index)
X_new_std = pd.concat([X_new_numeric_std, X_new_categoric], axis=1)
"""
# Change the order of the X_new_std 
X_new_std = X_new

' \n# Do the standardization for the numeric part \nscaler = StandardScaler() \nX_new_numeric_columns = X_new_numeric.columns\nX_new_numeric_index = X_new_numeric.index \nX_new_numeric_std = scaler.fit_transform(X_new_numeric)\nX_new_numeric_std = pd.DataFrame(X_new_numeric_std,\n                                 columns=X_new_numeric_columns, \n                                 index=X_new_numeric_index)\nX_new_std = pd.concat([X_new_numeric_std, X_new_categoric], axis=1)\n'

In [17]:
# Standardize X_MAASTRO 
# Set the columns_to_drop which are categorical 
columns_to_drop = [col for col in MAASTRO_new.columns if col in categorical_columns]

# Drop the columns if they exist in the DataFrame and save the numeric part in X_new_numeric
MAASTRO_new_categoric = MAASTRO_new[columns_to_drop]
MAASTRO_new_numeric = MAASTRO_new.drop(columns=columns_to_drop, inplace=False)
"""
# Do the standardization for the numeric part 
MAASTRO_new_numeric_columns = MAASTRO_new_numeric.columns

MAASTRO_new_numeric_columns = MAASTRO_new_numeric.columns
MAASTRO_new_numeric_index = MAASTRO_new_numeric.index 
MAASTRO_new_numeric_std = scaler.transform(MAASTRO_new_numeric)
MAASTRO_new_numeric_std = pd.DataFrame(MAASTRO_new_numeric_std,
                                 columns=MAASTRO_new_numeric_columns, 
                                 index=MAASTRO_new_numeric_index)
MAASTRO_new_std = pd.concat([MAASTRO_new_numeric_std, MAASTRO_new_categoric], axis=1)
 """
# Change the order of the X_new_std 
MAASTRO_new_std = MAASTRO_new

'\n# Do the standardization for the numeric part \nMAASTRO_new_numeric_columns = MAASTRO_new_numeric.columns\n\nMAASTRO_new_numeric_columns = MAASTRO_new_numeric.columns\nMAASTRO_new_numeric_index = MAASTRO_new_numeric.index \nMAASTRO_new_numeric_std = scaler.transform(MAASTRO_new_numeric)\nMAASTRO_new_numeric_std = pd.DataFrame(MAASTRO_new_numeric_std,\n                                 columns=MAASTRO_new_numeric_columns, \n                                 index=MAASTRO_new_numeric_index)\nMAASTRO_new_std = pd.concat([MAASTRO_new_numeric_std, MAASTRO_new_categoric], axis=1)\n '

In [18]:
X_new

,hpv_related,oropharynx,cavum_oris
0,0.0,1,0
1,0.0,0,0
2,0.0,0,1
3,0.0,0,0
4,0.0,0,0
...,...,...,...
134,1.0,1,0
135,1.0,1,0
136,1.0,1,0
137,1.0,1,0


In [19]:
X_new_std

,hpv_related,oropharynx,cavum_oris
0,0.0,1,0
1,0.0,0,0
2,0.0,0,1
3,0.0,0,0
4,0.0,0,0
...,...,...,...
134,1.0,1,0
135,1.0,1,0
136,1.0,1,0
137,1.0,1,0


In [20]:
MAASTRO_new

,hpv_related,oropharynx,cavum_oris
0,1,1,0
1,0,1,0
2,0,1,0
3,0,0,0
4,1,1,0
...,...,...,...
94,0,0,0
95,0,0,0
96,1,1,0
97,1,1,0


In [21]:
MAASTRO_new_std

,hpv_related,oropharynx,cavum_oris
0,1,1,0
1,0,1,0
2,0,1,0
3,0,0,0
4,1,1,0
...,...,...,...
94,0,0,0
95,0,0,0
96,1,1,0
97,1,1,0


# Modelling 

### 1. CoxPHSurvivalAnalysis

#### Train

In [22]:
# Setting the y format for skf below  
y = clinical_train[['DFS', 'event_DFS']]

# Running to optuna for hyperparameter tuning 
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class()
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = StandardScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxPHSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxPHSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-13 17:20:05,757] A new study created in memory with name: no-name-3fd8d94c-0803-484d-bc1e-dba7f3afc3e1


  0%|          | 0/1 [00:00<?, ?it/s]

[I 2024-04-13 17:20:12,957] A new study created in memory with name: no-name-96b0de4f-d3b7-48e8-9e16-6f8f36004f7e


Fold 1 C-index: 0.5936254980079682
Fold 2 C-index: 0.5755813953488372
Fold 3 C-index: 0.6893617021276596
Fold 4 C-index: 0.714828897338403
Fold 5 C-index: 0.5901287553648069
[I 2024-04-13 17:20:12,933] Trial 0 finished with value: 0.6327052496375349 and parameters: {}. Best is trial 0 with value: 0.6327052496375349.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.6327052496375349], datetime_start=datetime.datetime(2024, 4, 13, 17, 20, 5, 841735), datetime_complete=datetime.datetime(2024, 4, 13, 17, 20, 12, 933244), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.6327052496375349


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.23087000581287248
Fold 2 IBS: 0.23998071068865376
Fold 3 IBS: 0.1971849102359433
Fold 4 IBS: 0.235686306455164
Fold 5 IBS: 0.21379155522848114
[I 2024-04-13 17:20:13,311] Trial 0 finished with value: 0.22350269768422293 and parameters: {}. Best is trial 0 with value: 0.22350269768422293.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.22350269768422293], datetime_start=datetime.datetime(2024, 4, 13, 17, 20, 12, 997714), datetime_complete=datetime.datetime(2024, 4, 13, 17, 20, 13, 311354), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.22350269768422293


In [23]:
# Setting a dictionary to save the train results 
train_cindex = {} 
train_ibs = {} 

# Saving the values to the dictionary 
train_cindex['CoxPH'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxPH'] = np.round(study_ibs.best_value, 3)

In [24]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.633
train_ibs:  0.224


#### Test

In [25]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [26]:
# Test on MAASTRO 
cph = CoxPHSurvivalAnalysis()

cph.fit(X_new_std, y)

# Save C-index 
c_index = cph.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print('Concordance index:', c_index)

# Save IBS 
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in cph.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print('IBS score:', ibs)

CoxPHSurvivalAnalysis()

Concordance index: 0.55
IBS score: 0.235


In [27]:
# Setting a dictionary to save the test results 
test_cindex = {} 
test_ibs = {} 

In [28]:
# Saving the values to the dictionary 
test_cindex['CoxPH'] = c_index
test_ibs['CoxPH'] = ibs

### 2. CoxnetSurvivalAnalysis - Ridge 

#### Train

In [29]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

# Running to optuna for hyperparameter tuning 
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class(l1_ratio=0.0000001, 
                            fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = StandardScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-13 17:20:13,580] A new study created in memory with name: no-name-9b869c0f-75ae-4811-82e2-9f6725709552


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.5936254980079682
Fold 2 C-index: 0.5891472868217055


[I 2024-04-13 17:20:13,742] A new study created in memory with name: no-name-99bea0dd-2f86-4a2a-a2d6-2722cbf01711


Fold 3 C-index: 0.6361702127659574
Fold 4 C-index: 0.6901140684410646
Fold 5 C-index: 0.6030042918454935
[I 2024-04-13 17:20:13,733] Trial 0 finished with value: 0.6224122715764377 and parameters: {}. Best is trial 0 with value: 0.6224122715764377.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.6224122715764377], datetime_start=datetime.datetime(2024, 4, 13, 17, 20, 13, 618891), datetime_complete=datetime.datetime(2024, 4, 13, 17, 20, 13, 733099), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.6224122715764377


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.24724709767921887
Fold 2 IBS: 0.2320398826136388
Fold 3 IBS: 0.22898186436633558
Fold 4 IBS: 0.2419747678451801
Fold 5 IBS: 0.22939559031800452
[I 2024-04-13 17:20:13,965] Trial 0 finished with value: 0.23592784056447558 and parameters: {}. Best is trial 0 with value: 0.23592784056447558.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.23592784056447558], datetime_start=datetime.datetime(2024, 4, 13, 17, 20, 13, 777577), datetime_complete=datetime.datetime(2024, 4, 13, 17, 20, 13, 965400), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.23592784056447558


In [30]:
train_cindex['CoxRidge'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxRidge'] = np.round(study_ibs.best_value, 3)

In [31]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.622
train_ibs:  0.236


#### Test

In [32]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [33]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params['l1_ratio'] = 0.0000001
    best_params['fit_baseline_model']=True
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1e-07)

test_cindex : 0.653


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1e-07)

test_ibs:  0.229


In [34]:
# Saving the values to the dictionary 
test_cindex['CoxRidge'] = c_index
test_ibs['CoxRidge'] = ibs

### 3. CoxnetSurvivalAnalysis - Lasso

#### Train

In [35]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class(l1_ratio=1, 
                            fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = StandardScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-13 17:20:14,283] A new study created in memory with name: no-name-ec26436a-9889-4db2-b30e-cc7f073b9230


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.5936254980079682
Fold 2 C-index: 0.5755813953488372
Fold 3 C-index: 0.6893617021276596


[I 2024-04-13 17:20:14,753] A new study created in memory with name: no-name-bd98633b-4601-40b2-876d-34c6ae090373


Fold 4 C-index: 0.714828897338403
Fold 5 C-index: 0.5901287553648069
[I 2024-04-13 17:20:14,738] Trial 0 finished with value: 0.6327052496375349 and parameters: {}. Best is trial 0 with value: 0.6327052496375349.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.6327052496375349], datetime_start=datetime.datetime(2024, 4, 13, 17, 20, 14, 392651), datetime_complete=datetime.datetime(2024, 4, 13, 17, 20, 14, 737937), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.6327052496375349


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.23062952029645012
Fold 2 IBS: 0.23878423422778727
Fold 3 IBS: 0.1979953785766009
Fold 4 IBS: 0.23468273038135695
Fold 5 IBS: 0.21326774090755368
[I 2024-04-13 17:20:15,263] Trial 0 finished with value: 0.22307192087794978 and parameters: {}. Best is trial 0 with value: 0.22307192087794978.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.22307192087794978], datetime_start=datetime.datetime(2024, 4, 13, 17, 20, 14, 843961), datetime_complete=datetime.datetime(2024, 4, 13, 17, 20, 15, 263563), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.22307192087794978


In [36]:
train_cindex['CoxLasso'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxLasso'] = np.round(study_ibs.best_value, 3)

In [37]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.633
train_ibs:  0.223


#### Test 

In [38]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [39]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params['l1_ratio'] = 1
    best_params['fit_baseline_model']=True
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1)

test_cindex : 0.55


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1)

test_ibs:  0.233


In [40]:
# Saving the values to the dictionary 
test_cindex['CoxLasso'] = c_index
test_ibs['CoxLasso'] = ibs

### 4. CoxnetSurvivalAnalysis - ElasticNet

#### Train

In [41]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        l1_ratio = trial.suggest_float("l1_ratio", 0.0001, 1)
        
        # Create and fit survival model 
        model = model_class(l1_ratio=l1_ratio, 
                           fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = StandardScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-13 17:20:15,693] A new study created in memory with name: no-name-0ac634e7-7256-4168-a279-a92b0303ecee


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5936254980079682
Fold 2 C-index: 0.5755813953488372
Fold 3 C-index: 0.6893617021276596
Fold 4 C-index: 0.714828897338403
Fold 5 C-index: 0.5901287553648069
[I 2024-04-13 17:20:16,133] Trial 0 finished with value: 0.6327052496375349 and parameters: {'l1_ratio': 0.6964995386793018}. Best is trial 0 with value: 0.6327052496375349.
Fold 1 C-index: 0.5936254980079682
Fold 2 C-index: 0.5755813953488372
Fold 3 C-index: 0.6893617021276596
Fold 4 C-index: 0.6958174904942965
Fold 5 C-index: 0.5901287553648069
[I 2024-04-13 17:20:16,512] Trial 1 finished with value: 0.6289029682687136 and parameters: {'l1_ratio': 0.28621072101688444}. Best is trial 0 with value: 0.6327052496375349.
Fold 1 C-index: 0.5936254980079682
Fold 2 C-index: 0.5891472868217055
Fold 3 C-index: 0.6893617021276596
Fold 4 C-index: 0.6958174904942965
Fold 5 C-index: 0.592274678111588
[I 2024-04-13 17:20:16,712] Trial 2 finished with value: 0.6320453311126435 and parameters: {'l1_ratio': 0.22692876841884668}. B

Fold 3 C-index: 0.6893617021276596
Fold 4 C-index: 0.714828897338403
Fold 5 C-index: 0.5901287553648069
[I 2024-04-13 17:20:27,574] Trial 24 finished with value: 0.6327052496375349 and parameters: {'l1_ratio': 0.7602371709740536}. Best is trial 0 with value: 0.6327052496375349.
Fold 1 C-index: 0.5936254980079682
Fold 2 C-index: 0.5755813953488372
Fold 3 C-index: 0.6893617021276596
Fold 4 C-index: 0.714828897338403
Fold 5 C-index: 0.5901287553648069
[I 2024-04-13 17:20:28,264] Trial 25 finished with value: 0.6327052496375349 and parameters: {'l1_ratio': 0.891573855041803}. Best is trial 0 with value: 0.6327052496375349.
Fold 1 C-index: 0.5936254980079682
Fold 2 C-index: 0.5755813953488372
Fold 3 C-index: 0.6893617021276596
Fold 4 C-index: 0.714828897338403
Fold 5 C-index: 0.5901287553648069
[I 2024-04-13 17:20:28,875] Trial 26 finished with value: 0.6327052496375349 and parameters: {'l1_ratio': 0.4514399534035586}. Best is trial 0 with value: 0.6327052496375349.
Fold 1 C-index: 0.593625

Fold 2 C-index: 0.5755813953488372
Fold 3 C-index: 0.6893617021276596
Fold 4 C-index: 0.714828897338403
Fold 5 C-index: 0.5901287553648069
[I 2024-04-13 17:20:38,463] Trial 48 finished with value: 0.6327052496375349 and parameters: {'l1_ratio': 0.5033132688848968}. Best is trial 0 with value: 0.6327052496375349.
Fold 1 C-index: 0.5936254980079682
Fold 2 C-index: 0.5755813953488372
Fold 3 C-index: 0.6893617021276596
Fold 4 C-index: 0.714828897338403
Fold 5 C-index: 0.5901287553648069
[I 2024-04-13 17:20:39,057] Trial 49 finished with value: 0.6327052496375349 and parameters: {'l1_ratio': 0.355745098499332}. Best is trial 0 with value: 0.6327052496375349.
Fold 1 C-index: 0.5936254980079682
Fold 2 C-index: 0.5755813953488372
Fold 3 C-index: 0.6893617021276596
Fold 4 C-index: 0.714828897338403
Fold 5 C-index: 0.5901287553648069
[I 2024-04-13 17:20:39,548] Trial 50 finished with value: 0.6327052496375349 and parameters: {'l1_ratio': 0.8449632453631055}. Best is trial 0 with value: 0.6327052

...

  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 IBS: 0.2316547290662942
Fold 2 IBS: 0.23879236312745447
Fold 3 IBS: 0.19825571013048512
Fold 4 IBS: 0.2346684603917355
Fold 5 IBS: 0.21323103060583587
[I 2024-04-13 17:21:05,420] Trial 0 finished with value: 0.22332045866436104 and parameters: {'l1_ratio': 0.6964995386793018}. Best is trial 0 with value: 0.22332045866436104.
Fold 1 IBS: 0.2328215420107209
Fold 2 IBS: 0.2387376694592152
Fold 3 IBS: 0.19855864371824158
Fold 4 IBS: 0.23535239124869783
Fold 5 IBS: 0.21321676645113197
[I 2024-04-13 17:21:06,159] Trial 1 finished with value: 0.2237374025776015 and parameters: {'l1_ratio': 0.28621072101688444}. Best is trial 0 with value: 0.22332045866436104.
Fold 1 IBS: 0.23290719466305726
Fold 2 IBS: 0.22910469892979343
Fold 3 IBS: 0.19851761912519322
Fold 4 IBS: 0.23645133312318467
Fold 5 IBS: 0.22509043345132884
[I 2024-04-13 17:21:06,607] Trial 2 finished with value: 0.22441425585851152 and parameters: {'l1_ratio': 0.22692876841884668}. Best is trial 0 with value: 0.22332045866436

Fold 2 IBS: 0.23878224953918464
Fold 3 IBS: 0.19810697263118887
Fold 4 IBS: 0.23470141904586134
Fold 5 IBS: 0.21322019839910245
[I 2024-04-13 17:21:18,266] Trial 25 finished with value: 0.2231471001728039 and parameters: {'l1_ratio': 0.9106143729430608}. Best is trial 11 with value: 0.22307290116668219.
Fold 1 IBS: 0.23147019321082243
Fold 2 IBS: 0.23875695226908134
Fold 3 IBS: 0.19827506796402766
Fold 4 IBS: 0.23471962211441358
Fold 5 IBS: 0.21321402952933738
[I 2024-04-13 17:21:18,693] Trial 26 finished with value: 0.22328717301753645 and parameters: {'l1_ratio': 0.7417169602645222}. Best is trial 11 with value: 0.22307290116668219.
Fold 1 IBS: 0.23178762835502842
Fold 2 IBS: 0.23881660346673023
Fold 3 IBS: 0.19834944165203136
Fold 4 IBS: 0.23470201310790353
Fold 5 IBS: 0.2132448151903195
[I 2024-04-13 17:21:19,294] Trial 27 finished with value: 0.22338010035440262 and parameters: {'l1_ratio': 0.6494728799686162}. Best is trial 11 with value: 0.22307290116668219.
Fold 1 IBS: 0.231146

Fold 1 IBS: 0.23103357095404822
Fold 2 IBS: 0.23873849199860037
Fold 3 IBS: 0.1981519591601307
Fold 4 IBS: 0.2346730973202225
Fold 5 IBS: 0.2132514462936442
[I 2024-04-13 17:21:31,270] Trial 50 finished with value: 0.22316971314532918 and parameters: {'l1_ratio': 0.8786447363305766}. Best is trial 40 with value: 0.2230719959180285.
Fold 1 IBS: 0.23079344247298086
Fold 2 IBS: 0.23882983880964445
Fold 3 IBS: 0.1980560175062932
Fold 4 IBS: 0.23473334901576678
Fold 5 IBS: 0.21324134583800086
[I 2024-04-13 17:21:31,746] Trial 51 finished with value: 0.22313079872853722 and parameters: {'l1_ratio': 0.9495136153828908}. Best is trial 40 with value: 0.2230719959180285.
Fold 1 IBS: 0.23077282357670065
Fold 2 IBS: 0.2388373168564218
Fold 3 IBS: 0.1980483437764446
Fold 4 IBS: 0.23464655857618702
Fold 5 IBS: 0.21324469257518483
[I 2024-04-13 17:21:32,206] Trial 52 finished with value: 0.22310994707218773 and parameters: {'l1_ratio': 0.9556391205035538}. Best is trial 40 with value: 0.2230719959180

Fold 1 IBS: 0.2308833739728706
Fold 2 IBS: 0.23879879077117747
Fold 3 IBS: 0.19809001921386574
Fold 4 IBS: 0.23471207316639198
Fold 5 IBS: 0.21322759479315892
[I 2024-04-13 17:21:42,823] Trial 75 finished with value: 0.22314237038349294 and parameters: {'l1_ratio': 0.9232248440809541}. Best is trial 40 with value: 0.2230719959180285.
Fold 1 IBS: 0.23073384996338173
Fold 2 IBS: 0.23874357988596578
Fold 3 IBS: 0.19803387434188405
Fold 4 IBS: 0.23465645199769555
Fold 5 IBS: 0.21325097847391167
[I 2024-04-13 17:21:43,275] Trial 76 finished with value: 0.22308374693256777 and parameters: {'l1_ratio': 0.9673906281038791}. Best is trial 40 with value: 0.2230719959180285.
Fold 1 IBS: 0.23096554530535066
Fold 2 IBS: 0.23876579972169462
Fold 3 IBS: 0.19812381484611113
Fold 4 IBS: 0.2346908244723955
Fold 5 IBS: 0.2132627355802848
[I 2024-04-13 17:21:43,670] Trial 77 finished with value: 0.22316174398516733 and parameters: {'l1_ratio': 0.8983978427418612}. Best is trial 40 with value: 0.2230719959

In [42]:
train_cindex['CoxElastic'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxElastic'] = np.round(study_ibs.best_value, 3)

In [43]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.633
train_ibs:  0.223


#### Test

In [44]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [45]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    return model_class(**best_params, fit_baseline_model=True)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=0.6964995386793018)

test_cindex : 0.55


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=0.9997874437591321)

test_ibs:  0.233


In [46]:
# Saving the values to the dictionary 
test_cindex['CoxElastic'] = c_index
test_ibs['CoxElastic'] = ibs

### 5. Random Survival Forest

#### Train

In [47]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        oob_score = trial.suggest_categorical("oob_score", [True, False])
        max_samples = trial.suggest_float("max_samples", 0.1, 1.0) 
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None])
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        
        # Include warm_start for C-index optimization
        if metric == "c-index":
            warm_start = trial.suggest_categorical("warm_start", [True, False])
        else:
            warm_start = False  # Exclude warm_start for other metrics
        
        # Create and fit survival model 
        model = model_class(min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            max_leaf_nodes=max_leaf_nodes,
                            n_estimators=n_estimators, 
                            oob_score=oob_score,
                            warm_start=warm_start,
                            max_depth=max_depth,
                            max_features=max_features,
                            min_weight_fraction_leaf=min_weight_fraction_leaf, 
                            max_samples=max_samples, 
                            random_state=123)

        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])
            
            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(RandomSurvivalForest, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(RandomSurvivalForest, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-13 17:21:54,417] A new study created in memory with name: no-name-03d57495-0bbc-4ea9-a150-8984fb80bd2e


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5936254980079682
Fold 2 C-index: 0.6046511627906976
Fold 3 C-index: 0.6361702127659574
Fold 4 C-index: 0.6901140684410646
Fold 5 C-index: 0.6030042918454935
[I 2024-04-13 17:21:59,309] Trial 0 finished with value: 0.6255130467702362 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 7, 'min_samples_leaf': 5, 'max_depth': 12, 'n_estimators': 360, 'oob_score': False, 'max_samples': 0.7163467647263769, 'max_features': None, 'min_weight_fraction_leaf': 0.2192861223398122, 'warm_start': False}. Best is trial 0 with value: 0.6255130467702362.
Fold 1 C-index: 0.5936254980079682
Fold 2 C-index: 0.6027131782945736
Fold 3 C-index: 0.6361702127659574
Fold 4 C-index: 0.7072243346007605
Fold 5 C-index: 0.6072961373390557
[I 2024-04-13 17:22:03,374] Trial 1 finished with value: 0.6294058722016631 and parameters: {'min_samples_split': 16, 'max_leaf_nodes': 5, 'min_samples_leaf': 4, 'max_depth': 11, 'n_estimators': 266, 'oob_score': False, 'max_samples': 0.7520097923745717, 

Fold 3 C-index: 0.6361702127659574
Fold 4 C-index: 0.6901140684410646
Fold 5 C-index: 0.6030042918454935
[I 2024-04-13 17:22:51,825] Trial 15 finished with value: 0.6255130467702362 and parameters: {'min_samples_split': 19, 'max_leaf_nodes': 14, 'min_samples_leaf': 3, 'max_depth': 7, 'n_estimators': 170, 'oob_score': True, 'max_samples': 0.834513235692611, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.1695220400209786, 'warm_start': True}. Best is trial 14 with value: 0.6400441700740036.
Fold 1 C-index: 0.5936254980079682
Fold 2 C-index: 0.6046511627906976
Fold 3 C-index: 0.6361702127659574
Fold 4 C-index: 0.6901140684410646
Fold 5 C-index: 0.6030042918454935
[I 2024-04-13 17:22:52,822] Trial 16 finished with value: 0.6255130467702362 and parameters: {'min_samples_split': 12, 'max_leaf_nodes': 17, 'min_samples_leaf': 1, 'max_depth': 7, 'n_estimators': 122, 'oob_score': True, 'max_samples': 0.5394490956397513, 'max_features': 'sqrt', 'min_weight_fraction_leaf': 0.131533631322503

Fold 1 C-index: 0.5936254980079682
Fold 2 C-index: 0.6046511627906976
Fold 3 C-index: 0.6361702127659574
Fold 4 C-index: 0.6901140684410646
Fold 5 C-index: 0.6030042918454935
[I 2024-04-13 17:23:13,963] Trial 30 finished with value: 0.6255130467702362 and parameters: {'min_samples_split': 11, 'max_leaf_nodes': 19, 'min_samples_leaf': 4, 'max_depth': 5, 'n_estimators': 87, 'oob_score': True, 'max_samples': 0.5224151807092315, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.18579705931910293, 'warm_start': True}. Best is trial 14 with value: 0.6400441700740036.
Fold 1 C-index: 0.5936254980079682
Fold 2 C-index: 0.6027131782945736
Fold 3 C-index: 0.6893617021276596
Fold 4 C-index: 0.7072243346007605
Fold 5 C-index: 0.6072961373390557
[I 2024-04-13 17:23:15,757] Trial 31 finished with value: 0.6400441700740036 and parameters: {'min_samples_split': 10, 'max_leaf_nodes': 17, 'min_samples_leaf': 7, 'max_depth': 2, 'n_estimators': 234, 'oob_score': True, 'max_samples': 0.4804335394063386

...

Fold 1 IBS: 0.22938449208506656
Fold 2 IBS: 0.22753082201660127
Fold 3 IBS: 0.20896337795479314
Fold 4 IBS: 0.21837680226854744
Fold 5 IBS: 0.21213517485483555
[I 2024-04-13 17:29:51,847] Trial 32 finished with value: 0.21927813383596878 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 14, 'min_samples_leaf': 2, 'max_depth': 14, 'n_estimators': 310, 'oob_score': True, 'max_samples': 0.9502203520505428, 'max_features': None, 'min_weight_fraction_leaf': 0.3660826370427841}. Best is trial 13 with value: 0.2191628532255277.
Fold 1 IBS: 0.2293305571674784
Fold 2 IBS: 0.2249056666416748
Fold 3 IBS: 0.21231192754080194
Fold 4 IBS: 0.2211259092827611
Fold 5 IBS: 0.2121489087663462
[I 2024-04-13 17:29:59,643] Trial 33 finished with value: 0.2199645938798125 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 14, 'min_samples_leaf': 4, 'max_depth': 14, 'n_estimators': 308, 'oob_score': True, 'max_samples': 0.9355742306337576, 'max_features': None, 'min_weight_fraction_leaf': 0

Fold 1 IBS: 0.22927545380946876
Fold 2 IBS: 0.2276310370401933
Fold 3 IBS: 0.20885917650739413
Fold 4 IBS: 0.21808141368878975
Fold 5 IBS: 0.21215244703182187
[I 2024-04-13 17:31:46,431] Trial 48 finished with value: 0.21919990561553354 and parameters: {'min_samples_split': 4, 'max_leaf_nodes': 17, 'min_samples_leaf': 15, 'max_depth': 17, 'n_estimators': 408, 'oob_score': False, 'max_samples': 0.9576806605170014, 'max_features': None, 'min_weight_fraction_leaf': 0.365900430347092}. Best is trial 13 with value: 0.2191628532255277.
Fold 1 IBS: 0.23920717494427576
Fold 2 IBS: 0.22840267598061034
Fold 3 IBS: 0.2254393339269583
Fold 4 IBS: 0.23467057175960346
Fold 5 IBS: 0.221637000000393
[I 2024-04-13 17:31:54,531] Trial 49 finished with value: 0.22987135132236816 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 10, 'min_samples_leaf': 17, 'max_depth': 19, 'n_estimators': 403, 'oob_score': False, 'max_samples': 0.6731598118016386, 'max_features': None, 'min_weight_fraction_leaf':

Fold 1 IBS: 0.2460267923298924
Fold 2 IBS: 0.2322217659737921
Fold 3 IBS: 0.22995325803591157
Fold 4 IBS: 0.24129229641724947
Fold 5 IBS: 0.23023978177796017
[I 2024-04-13 17:34:07,950] Trial 64 finished with value: 0.23594677890696114 and parameters: {'min_samples_split': 4, 'max_leaf_nodes': 17, 'min_samples_leaf': 12, 'max_depth': 16, 'n_estimators': 401, 'oob_score': False, 'max_samples': 0.26967029716304014, 'max_features': None, 'min_weight_fraction_leaf': 0.33799253233948007}. Best is trial 52 with value: 0.21912978414936357.
Fold 1 IBS: 0.23418590920687307
Fold 2 IBS: 0.2318005678098793
Fold 3 IBS: 0.21076727004417795
Fold 4 IBS: 0.21591612664228851
Fold 5 IBS: 0.21267569020316077
[I 2024-04-13 17:34:16,470] Trial 65 finished with value: 0.22106911278127592 and parameters: {'min_samples_split': 7, 'max_leaf_nodes': 15, 'min_samples_leaf': 15, 'max_depth': 16, 'n_estimators': 461, 'oob_score': False, 'max_samples': 0.9287628030071836, 'max_features': None, 'min_weight_fraction_l

Fold 5 IBS: 0.22999822825894972
[I 2024-04-13 17:36:38,194] Trial 79 finished with value: 0.23593860779874168 and parameters: {'min_samples_split': 13, 'max_leaf_nodes': 15, 'min_samples_leaf': 4, 'max_depth': 15, 'n_estimators': 473, 'oob_score': False, 'max_samples': 0.12074719353470553, 'max_features': None, 'min_weight_fraction_leaf': 0.44663557696688005}. Best is trial 52 with value: 0.21912978414936357.
Fold 1 IBS: 0.23023691170508595
Fold 2 IBS: 0.22831549805589474
Fold 3 IBS: 0.2085077533226
Fold 4 IBS: 0.2170371467540854
Fold 5 IBS: 0.21209842650340235
[I 2024-04-13 17:36:54,019] Trial 80 finished with value: 0.21923914726821367 and parameters: {'min_samples_split': 20, 'max_leaf_nodes': 19, 'min_samples_leaf': 10, 'max_depth': 17, 'n_estimators': 487, 'oob_score': True, 'max_samples': 0.8362923410326437, 'max_features': None, 'min_weight_fraction_leaf': 0.3136324910319987}. Best is trial 52 with value: 0.21912978414936357.
Fold 1 IBS: 0.23023691170508595
Fold 2 IBS: 0.2283154

Fold 1 IBS: 0.22909260836646478
Fold 2 IBS: 0.22503544485507979
Fold 3 IBS: 0.2120598829609088
Fold 4 IBS: 0.2214716876773108
Fold 5 IBS: 0.2120467826247585
[I 2024-04-13 17:40:02,851] Trial 95 finished with value: 0.21994128129690454 and parameters: {'min_samples_split': 19, 'max_leaf_nodes': 18, 'min_samples_leaf': 8, 'max_depth': 19, 'n_estimators': 487, 'oob_score': True, 'max_samples': 0.9807024729491963, 'max_features': None, 'min_weight_fraction_leaf': 0.4007304758915286}. Best is trial 92 with value: 0.21906389241461235.
Fold 1 IBS: 0.22894596533673903
Fold 2 IBS: 0.22806686188441844
Fold 3 IBS: 0.20882278342889127
Fold 4 IBS: 0.21784782165652053
Fold 5 IBS: 0.21215769756945996
[I 2024-04-13 17:40:17,983] Trial 96 finished with value: 0.21916822597520583 and parameters: {'min_samples_split': 20, 'max_leaf_nodes': 18, 'min_samples_leaf': 10, 'max_depth': 20, 'n_estimators': 477, 'oob_score': True, 'max_samples': 0.9808378834140379, 'max_features': None, 'min_weight_fraction_leaf

In [48]:
train_cindex['Randomsurvivalforest'] = np.round(study_cindex.best_value, 3)
train_ibs['Randomsurvivalforest'] = np.round(study_ibs.best_value, 3)

In [49]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.64
train_ibs:  0.219


#### Test

In [50]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))
    
y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [51]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(RandomSurvivalForest, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex: ", c_index)

# Set the best model 
best_model_ibs = create_best_model(RandomSurvivalForest, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

RandomSurvivalForest(max_depth=7, max_features='log2', max_leaf_nodes=13,
                     max_samples=0.8364130211722378, min_samples_split=12,
                     min_weight_fraction_leaf=0.1584235940276062,
                     n_estimators=192, oob_score=True, random_state=123,
                     warm_start=True)

test_cindex:  0.653


RandomSurvivalForest(max_depth=18, max_features=None, max_leaf_nodes=19,
                     max_samples=0.9732130654966473, min_samples_leaf=10,
                     min_samples_split=20,
                     min_weight_fraction_leaf=0.36798680886807317,
                     n_estimators=482, oob_score=True, random_state=123)

test_ibs:  0.221


In [52]:
# Saving the values to the dictionary 
test_cindex['Randomsurvivalforest'] = c_index
test_ibs['Randomsurvivalforest'] = ibs

### 6. ExtraSurvivalTrees

#### Train

In [53]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters 
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        oob_score = trial.suggest_categorical("oob_score", [True, False])
        warm_start = trial.suggest_categorical("warm_start", [True, False])
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None, 0.1, 1])
        max_samples = trial.suggest_float("max_samples", 0.1, 1.0) 
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        
        # Include warm_start for C-index optimization
        if metric == "c-index":
            warm_start = trial.suggest_categorical("warm_start", [True, False])
        else:
            warm_start = False  # Exclude warm_start for other metrics

        
        # Create and fit survival model 
        model = model_class(min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            max_leaf_nodes=max_leaf_nodes,
                            n_estimators=n_estimators, 
                            oob_score=oob_score, 
                            max_features=max_features, 
                            warm_start=warm_start, 
                            max_samples=max_samples,
                            min_weight_fraction_leaf=min_weight_fraction_leaf, 
                            max_depth=max_depth, 
                            random_state=123) 
        
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(ExtraSurvivalTrees, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(ExtraSurvivalTrees, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-13 17:41:09,361] A new study created in memory with name: no-name-92961422-218c-4a9f-9f55-f8068cb451be


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5936254980079682
Fold 2 C-index: 0.6027131782945736
Fold 3 C-index: 0.6893617021276596
Fold 4 C-index: 0.7072243346007605
Fold 5 C-index: 0.6072961373390557
[I 2024-04-13 17:41:11,587] Trial 0 finished with value: 0.6400441700740036 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 7, 'min_samples_leaf': 5, 'max_depth': 12, 'n_estimators': 360, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.7641958651588321, 'min_weight_fraction_leaf': 0.09124586522674999}. Best is trial 0 with value: 0.6400441700740036.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-13 17:41:16,947] Trial 1 finished with value: 0.5 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 12, 'min_samples_leaf': 11, 'max_depth': 13, 'n_estimators': 425, 'oob_score': True, 'warm_start': True, 'max_features': None, 'max_samples': 0.4877764869966794, 'min_weight_fraction_leaf': 0.2468425488251531

Fold 5 C-index: 0.6072961373390557
[I 2024-04-13 17:42:23,187] Trial 15 finished with value: 0.6400441700740036 and parameters: {'min_samples_split': 17, 'max_leaf_nodes': 9, 'min_samples_leaf': 5, 'max_depth': 9, 'n_estimators': 402, 'oob_score': False, 'warm_start': True, 'max_features': 'sqrt', 'max_samples': 0.6880212408103346, 'min_weight_fraction_leaf': 0.0788882352604838}. Best is trial 0 with value: 0.6400441700740036.
Fold 1 C-index: 0.5936254980079682
Fold 2 C-index: 0.6046511627906976
Fold 3 C-index: 0.6361702127659574
Fold 4 C-index: 0.6901140684410646
Fold 5 C-index: 0.6030042918454935
[I 2024-04-13 17:42:24,483] Trial 16 finished with value: 0.6255130467702362 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 17, 'min_samples_leaf': 7, 'max_depth': 20, 'n_estimators': 175, 'oob_score': False, 'warm_start': True, 'max_features': 'auto', 'max_samples': 0.8750346354626457, 'min_weight_fraction_leaf': 0.4093818399278544}. Best is trial 0 with value: 0.640044170074003

Fold 1 C-index: 0.5936254980079682
Fold 2 C-index: 0.6027131782945736
Fold 3 C-index: 0.6893617021276596
Fold 4 C-index: 0.7072243346007605
Fold 5 C-index: 0.6072961373390557
[I 2024-04-13 17:43:18,123] Trial 30 finished with value: 0.6400441700740036 and parameters: {'min_samples_split': 12, 'max_leaf_nodes': 3, 'min_samples_leaf': 6, 'max_depth': 15, 'n_estimators': 309, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.4344347067743851, 'min_weight_fraction_leaf': 0.0387114670088218}. Best is trial 0 with value: 0.6400441700740036.
Fold 1 C-index: 0.5936254980079682
Fold 2 C-index: 0.6027131782945736
Fold 3 C-index: 0.6893617021276596
Fold 4 C-index: 0.7072243346007605
Fold 5 C-index: 0.6072961373390557
[I 2024-04-13 17:43:21,701] Trial 31 finished with value: 0.6400441700740036 and parameters: {'min_samples_split': 14, 'max_leaf_nodes': 6, 'min_samples_leaf': 6, 'max_depth': 9, 'n_estimators': 500, 'oob_score': False, 'warm_start': True, 'max_features

...

Fold 1 IBS: 0.23399276619717632
Fold 2 IBS: 0.23002127894114327
Fold 3 IBS: 0.20843753875222787
Fold 4 IBS: 0.2160171050438523
Fold 5 IBS: 0.21248159379191128
[I 2024-04-13 17:55:31,817] Trial 30 finished with value: 0.2201900565452622 and parameters: {'min_samples_split': 7, 'max_leaf_nodes': 20, 'min_samples_leaf': 12, 'max_depth': 9, 'n_estimators': 287, 'oob_score': False, 'warm_start': False, 'max_features': None, 'max_samples': 0.8382504072008882, 'min_weight_fraction_leaf': 0.2917886569515737}. Best is trial 30 with value: 0.2201900565452622.
Fold 1 IBS: 0.23076757571589473
Fold 2 IBS: 0.22736450708093134
Fold 3 IBS: 0.20888035387329829
Fold 4 IBS: 0.21707897025702066
Fold 5 IBS: 0.2122428672001695
[I 2024-04-13 17:55:44,684] Trial 31 finished with value: 0.2192668548254629 and parameters: {'min_samples_split': 8, 'max_leaf_nodes': 20, 'min_samples_leaf': 12, 'max_depth': 9, 'n_estimators': 279, 'oob_score': False, 'warm_start': False, 'max_features': None, 'max_samples': 0.8304

Fold 1 IBS: 0.23270396069165322
Fold 2 IBS: 0.2247705563008025
Fold 3 IBS: 0.21879972765768144
Fold 4 IBS: 0.22790183002628017
Fold 5 IBS: 0.21399863217460155
[I 2024-04-13 17:57:56,764] Trial 45 finished with value: 0.22363494137020376 and parameters: {'min_samples_split': 12, 'max_leaf_nodes': 19, 'min_samples_leaf': 19, 'max_depth': 8, 'n_estimators': 227, 'oob_score': True, 'warm_start': False, 'max_features': None, 'max_samples': 0.9009043234203283, 'min_weight_fraction_leaf': 0.3924440977020272}. Best is trial 31 with value: 0.2192668548254629.
Fold 1 IBS: 0.23543434584536324
Fold 2 IBS: 0.22537115951414122
Fold 3 IBS: 0.22058241697244674
Fold 4 IBS: 0.23241260728025231
Fold 5 IBS: 0.21519591350899975
[I 2024-04-13 17:58:03,527] Trial 46 finished with value: 0.22579928862424067 and parameters: {'min_samples_split': 10, 'max_leaf_nodes': 16, 'min_samples_leaf': 20, 'max_depth': 10, 'n_estimators': 125, 'oob_score': True, 'warm_start': False, 'max_features': None, 'max_samples': 0.

Fold 1 IBS: 0.23666005092979583
Fold 2 IBS: 0.22960088648063198
Fold 3 IBS: 0.21026693389077333
Fold 4 IBS: 0.21605391871068125
Fold 5 IBS: 0.21236821700846553
[I 2024-04-13 18:00:35,959] Trial 60 finished with value: 0.2209900014040696 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 14, 'min_samples_leaf': 14, 'max_depth': 10, 'n_estimators': 74, 'oob_score': True, 'warm_start': False, 'max_features': None, 'max_samples': 0.9576577192543246, 'min_weight_fraction_leaf': 0.3073603256015917}. Best is trial 31 with value: 0.2192668548254629.
Fold 1 IBS: 0.23135626316866256
Fold 2 IBS: 0.22645553110407532
Fold 3 IBS: 0.20966122609641344
Fold 4 IBS: 0.21984941434508204
Fold 5 IBS: 0.21202533188860895
[I 2024-04-13 18:00:46,898] Trial 61 finished with value: 0.21986955332056848 and parameters: {'min_samples_split': 7, 'max_leaf_nodes': 17, 'min_samples_leaf': 16, 'max_depth': 6, 'n_estimators': 182, 'oob_score': True, 'warm_start': False, 'max_features': None, 'max_samples': 0.992

Fold 1 IBS: 0.2395442223429733
Fold 2 IBS: 0.22912945601230464
Fold 3 IBS: 0.22469665996463767
Fold 4 IBS: 0.23655575742859675
Fold 5 IBS: 0.2229792723639496
[I 2024-04-13 18:03:10,011] Trial 75 finished with value: 0.23058107362249242 and parameters: {'min_samples_split': 8, 'max_leaf_nodes': 17, 'min_samples_leaf': 16, 'max_depth': 9, 'n_estimators': 243, 'oob_score': True, 'warm_start': False, 'max_features': 'sqrt', 'max_samples': 0.8260830230779004, 'min_weight_fraction_leaf': 0.36247953138659794}. Best is trial 31 with value: 0.2192668548254629.
Fold 1 IBS: 0.2370931245618715
Fold 2 IBS: 0.22434008390481244
Fold 3 IBS: 0.2177837484984773
Fold 4 IBS: 0.22722248718170873
Fold 5 IBS: 0.21608148066710028
[I 2024-04-13 18:03:18,033] Trial 76 finished with value: 0.22450418496279406 and parameters: {'min_samples_split': 10, 'max_leaf_nodes': 12, 'min_samples_leaf': 12, 'max_depth': 10, 'n_estimators': 208, 'oob_score': True, 'warm_start': False, 'max_features': 1, 'max_samples': 0.9353

Fold 1 IBS: 0.23800441993348706
Fold 2 IBS: 0.22767850269903261
Fold 3 IBS: 0.2237206426833651
Fold 4 IBS: 0.23428835882497379
Fold 5 IBS: 0.2187778931045527
[I 2024-04-13 18:04:31,619] Trial 90 finished with value: 0.22849396344908227 and parameters: {'min_samples_split': 10, 'max_leaf_nodes': 20, 'min_samples_leaf': 10, 'max_depth': 13, 'n_estimators': 302, 'oob_score': False, 'warm_start': True, 'max_features': None, 'max_samples': 0.7414077144426384, 'min_weight_fraction_leaf': 0.3433264918658795}. Best is trial 31 with value: 0.2192668548254629.
Fold 1 IBS: 0.23257608160494517
Fold 2 IBS: 0.22777477042423552
Fold 3 IBS: 0.2089356907749039
Fold 4 IBS: 0.2174045202478247
Fold 5 IBS: 0.21222141483842463
[I 2024-04-13 18:04:40,257] Trial 91 finished with value: 0.2197824955780668 and parameters: {'min_samples_split': 9, 'max_leaf_nodes': 20, 'min_samples_leaf': 19, 'max_depth': 11, 'n_estimators': 133, 'oob_score': True, 'warm_start': True, 'max_features': None, 'max_samples': 0.86255

In [54]:
train_cindex['ExtraSurvivalTrees'] = np.round(study_cindex.best_value, 3)
train_ibs['ExtraSurvivalTrees'] = np.round(study_ibs.best_value, 3)

In [55]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.64
train_ibs:  0.219


#### Test

In [56]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [57]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(ExtraSurvivalTrees, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(ExtraSurvivalTrees, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

ExtraSurvivalTrees(max_depth=12, max_features='log2', max_leaf_nodes=7,
                   max_samples=0.7641958651588321, min_samples_leaf=5,
                   min_samples_split=15,
                   min_weight_fraction_leaf=0.09124586522674999,
                   n_estimators=360, random_state=123, warm_start=True)

C-index score: 0.654


ExtraSurvivalTrees(max_depth=9, max_features=None, max_leaf_nodes=20,
                   max_samples=0.8304907299557232, min_samples_leaf=12,
                   min_samples_split=8,
                   min_weight_fraction_leaf=0.3083299613298841,
                   n_estimators=279, random_state=123)

IBS: 0.222


In [58]:
# Saving the values to the dictionary 
test_cindex['ExtraSurvivalTrees'] = c_index
test_ibs['ExtraSurvivalTrees'] = ibs

### 7. GradientBoostingSurvivalAnalysis

#### Train

In [59]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

# Running to optuna for hyperparameter tuning
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        subsample = trial.suggest_float("subsample", 0.1, 1)
        learning_rate = trial.suggest_float("learning_rate", 0.001, 0.1)
        dropout_rate = trial.suggest_float("dropout_rate", 0.1, 1)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        criterion = trial.suggest_categorical('criterion', ['friedman_mse', 'squared_error'])
        ccp_alpha = trial.suggest_float("ccp_alpha", 0.0, 10)
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None, 0.1, 1])
        min_impurity_decrease = trial.suggest_loguniform('min_impurity_decrease', 1e-7, 1e-1)
        validation_fraction = trial.suggest_float("validation_fraction", 0.0, 1.0)
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        
        # Create and fit survival model 
        model = model_class(subsample=subsample,
                            learning_rate=learning_rate,
                            dropout_rate=dropout_rate,
                            n_estimators=n_estimators,
                            ccp_alpha=ccp_alpha, 
                            criterion=criterion,
                            min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            min_weight_fraction_leaf=min_weight_fraction_leaf,
                            max_depth=max_depth,
                            max_features=max_features,
                            max_leaf_nodes=max_leaf_nodes, 
                            min_impurity_decrease=min_impurity_decrease,
                            validation_fraction=validation_fraction, 
                            random_state=123)
        
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(GradientBoostingSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(GradientBoostingSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-13 18:06:07,464] A new study created in memory with name: no-name-a0c92d7a-b6ab-4dd8-a18b-c3eb35a71b0b


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-13 18:07:09,404] Trial 0 finished with value: 0.5 and parameters: {'subsample': 0.7268222670380755, 'learning_rate': 0.02932779416008757, 'dropout_rate': 0.3041663082077828, 'n_estimators': 276, 'criterion': 'friedman_mse', 'ccp_alpha': 9.807641983846155, 'min_weight_fraction_leaf': 0.34241486929243165, 'max_features': None, 'min_impurity_decrease': 2.4449249473284515e-05, 'validation_fraction': 0.7379954057320357, 'min_samples_split': 5, 'max_leaf_nodes': 5, 'min_samples_leaf': 11, 'max_depth': 11}. Best is trial 0 with value: 0.5.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-13 18:07:45,401] Trial 1 finished with value: 0.5 and parameters: {'subsample': 0.6709608626961889, 'learning_rate': 0.08509374761370117, 'dropout_rate': 0.7520097923745717, 'n_estimators': 306, 'criterion': 'friedman_mse', 'ccp_alpha': 3.

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-13 18:26:09,653] Trial 13 finished with value: 0.5 and parameters: {'subsample': 0.8386796524426539, 'learning_rate': 0.046734492485875676, 'dropout_rate': 0.4821375662037144, 'n_estimators': 402, 'criterion': 'squared_error', 'ccp_alpha': 1.5696007313501796, 'min_weight_fraction_leaf': 0.18684147934268416, 'max_features': 'log2', 'min_impurity_decrease': 1.5044881127471587e-06, 'validation_fraction': 0.8166356053932342, 'min_samples_split': 16, 'max_leaf_nodes': 19, 'min_samples_leaf': 16, 'max_depth': 4}. Best is trial 12 with value: 0.6214070332403031.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-13 18:28:21,214] Trial 14 finished with value: 0.5 and parameters: {'subsample': 0.33235389014851724, 'learning_rate': 0.04522573411670834, 'dropout_rate': 0.2712811374536856, 'n_estimators': 405, 'criterion': 'squar

Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-13 18:49:47,852] Trial 25 finished with value: 0.5 and parameters: {'subsample': 0.7565257046780437, 'learning_rate': 0.01188344684416992, 'dropout_rate': 0.3433523077110169, 'n_estimators': 318, 'criterion': 'squared_error', 'ccp_alpha': 2.063559212730723, 'min_weight_fraction_leaf': 0.25401273903421573, 'max_features': 'auto', 'min_impurity_decrease': 5.773435946665558e-07, 'validation_fraction': 0.8062268184400477, 'min_samples_split': 16, 'max_leaf_nodes': 18, 'min_samples_leaf': 5, 'max_depth': 3}. Best is trial 22 with value: 0.6366905799047093.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-13 18:52:07,245] Trial 26 finished with value: 0.5 and parameters: {'subsample': 0.8922404482621683, 'learning_rate': 0.011585260292674536, 'dropout_rate': 0.2527000999648632, 'n_estimators': 446, 'criterion': 'friedman_mse', 'ccp_alpha': 0.11820935501930148, 'min_weight_fraction

...

Fold 5 IBS: 0.2293955930480925
[I 2024-04-13 21:15:04,304] Trial 44 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.5628832155203615, 'learning_rate': 0.05692286295358468, 'dropout_rate': 0.683156188511181, 'n_estimators': 343, 'criterion': 'friedman_mse', 'ccp_alpha': 9.07255185322531, 'min_weight_fraction_leaf': 0.21518215373385774, 'max_features': None, 'min_impurity_decrease': 0.00078866474186123, 'validation_fraction': 0.8420815814864784, 'min_samples_split': 3, 'max_leaf_nodes': 11, 'min_samples_leaf': 17, 'max_depth': 10}. Best is trial 9 with value: 0.2356612304106684.
Fold 1 IBS: 0.24724710044658998
Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977708
Fold 4 IBS: 0.24197477145927118
Fold 5 IBS: 0.22939559304809248
[I 2024-04-13 21:15:14,446] Trial 45 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.9436582561003044, 'learning_rate': 0.07482848784358435, 'dropout_rate': 0.8595747969120033, 'n_estimators': 234, 'criteri

Fold 1 IBS: 0.24724710044658998
Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.2293955930480925
[I 2024-04-13 21:29:05,913] Trial 56 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.48190513355031883, 'learning_rate': 0.030884976740502446, 'dropout_rate': 0.19804020971050568, 'n_estimators': 358, 'criterion': 'friedman_mse', 'ccp_alpha': 1.843725001532687, 'min_weight_fraction_leaf': 0.3039444677981973, 'max_features': None, 'min_impurity_decrease': 0.0014301881576558163, 'validation_fraction': 0.5010775717552909, 'min_samples_split': 3, 'max_leaf_nodes': 3, 'min_samples_leaf': 9, 'max_depth': 13}. Best is trial 51 with value: 0.23245596848737357.
Fold 1 IBS: 0.24492508105172847
Fold 2 IBS: 0.23019092753434325
Fold 3 IBS: 0.22573510086260665
Fold 4 IBS: 0.2391183990980785
Fold 5 IBS: 0.22690712026893198
[I 2024-04-13 21:30:03,557] Trial 57 finished with value: 0.23337532576313777 and parameters: {'su

Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977708
Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.22939559304809248
[I 2024-04-13 21:39:22,904] Trial 67 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.3366291337803353, 'learning_rate': 0.03684138433344805, 'dropout_rate': 0.1328866027244634, 'n_estimators': 319, 'criterion': 'friedman_mse', 'ccp_alpha': 1.0074500878262913, 'min_weight_fraction_leaf': 0.40650496560519606, 'max_features': None, 'min_impurity_decrease': 0.00018562329660606072, 'validation_fraction': 0.1720340754780943, 'min_samples_split': 2, 'max_leaf_nodes': 3, 'min_samples_leaf': 12, 'max_depth': 16}. Best is trial 63 with value: 0.2324212552964143.
Fold 1 IBS: 0.24724710044658998
Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.24197477145927118
Fold 5 IBS: 0.2293955930480925
[I 2024-04-13 21:40:05,394] Trial 68 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.4450176007008625, '

Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.22939559304809248
[I 2024-04-13 21:50:56,684] Trial 78 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.38353946327089455, 'learning_rate': 0.04627201780375246, 'dropout_rate': 0.10185034849810722, 'n_estimators': 296, 'criterion': 'friedman_mse', 'ccp_alpha': 1.4077536708792413, 'min_weight_fraction_leaf': 0.37826406364062987, 'max_features': None, 'min_impurity_decrease': 4.870590304744988e-05, 'validation_fraction': 0.3211844074125865, 'min_samples_split': 6, 'max_leaf_nodes': 5, 'min_samples_leaf': 12, 'max_depth': 10}. Best is trial 63 with value: 0.2324212552964143.
Fold 1 IBS: 0.24551697513707202
Fold 2 IBS: 0.2303259588455767
Fold 3 IBS: 0.22662101843621418
Fold 4 IBS: 0.2394168423767029
Fold 5 IBS: 0.22718986074333442
[I 2024-04-13 22:03:56,331] Trial 79 finished with value: 0.23381413110778007 and parameters: {'subsample': 0.4449351033565357, 'learning_rate': 0.05447197006738

Fold 4 IBS: 0.23880006092678727
Fold 5 IBS: 0.22633462320742237
[I 2024-04-13 22:22:39,095] Trial 89 finished with value: 0.233058648186594 and parameters: {'subsample': 0.2799652441361268, 'learning_rate': 0.041598274959482893, 'dropout_rate': 0.21483185460780047, 'n_estimators': 350, 'criterion': 'friedman_mse', 'ccp_alpha': 0.019001072018239806, 'min_weight_fraction_leaf': 0.27872060184788355, 'max_features': None, 'min_impurity_decrease': 0.01806689386438656, 'validation_fraction': 0.5765746087299453, 'min_samples_split': 3, 'max_leaf_nodes': 6, 'min_samples_leaf': 10, 'max_depth': 15}. Best is trial 63 with value: 0.2324212552964143.
Fold 1 IBS: 0.24724710044658998
Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.2293955930480925
[I 2024-04-13 22:23:15,120] Trial 90 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.3053150575440778, 'learning_rate': 0.041388830250024856, 'dropout_rate': 0.226362271

In [60]:
train_cindex['GradientBoosting'] = np.round(study_cindex.best_value, 3)
train_ibs['GradientBoosting'] = np.round(study_ibs.best_value, 3)

In [61]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.637
train_ibs:  0.232


#### Test

In [62]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [63]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(GradientBoostingSurvivalAnalysis, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(GradientBoostingSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

GradientBoostingSurvivalAnalysis(ccp_alpha=0.0339977959383996,
                                 criterion='squared_error',
                                 dropout_rate=0.2075412325353082,
                                 learning_rate=0.010706280861824496,
                                 max_features='auto', max_leaf_nodes=17,
                                 min_impurity_decrease=3.3602815261835675e-07,
                                 min_samples_leaf=13, min_samples_split=18,
                                 min_weight_fraction_leaf=0.4472167339801619,
                                 n_estimators=445, random_state=123,
                                 subsample=0.9030031356045858,
                                 validation_fraction=0.9350158433232643)

C-index score: 0.598


GradientBoostingSurvivalAnalysis(ccp_alpha=0.017414333014397508,
                                 dropout_rate=0.13345731559321464,
                                 learning_rate=0.048781242977699715,
                                 max_depth=15, max_leaf_nodes=3,
                                 min_impurity_decrease=0.0005725956539451149,
                                 min_samples_leaf=9,
                                 min_weight_fraction_leaf=0.4053224848913279,
                                 n_estimators=356, random_state=123,
                                 subsample=0.38928742192747917,
                                 validation_fraction=0.31265661774106646)

IBS: 0.225


In [64]:
# Saving the values to the dictionary 
test_cindex['GradientBoosting'] = c_index
test_ibs['GradientBoosting'] = ibs

### 8. ComponentwiseGradientBoostingSurvivalAnalysis

#### Train

In [65]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        subsample = trial.suggest_float("subsample", 0.1, 1)
        dropout_rate = trial.suggest_float("dropout_rate", 0.1, 1)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        learning_rate = trial.suggest_float("learning_rate", 0.001, 0.1)
        
        # Create and fit survival model 
        model = model_class(subsample=subsample,
                            dropout_rate=dropout_rate,
                            n_estimators=n_estimators,
                            learning_rate=learning_rate,
                            random_state=123)
                
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = StandardScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective


# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(ComponentwiseGradientBoostingSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best hyperparameters for C-index: \n", study_cindex.best_params)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(ComponentwiseGradientBoostingSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best hyperparameters for IBS: \n", study_ibs.best_params)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-13 23:33:36,401] A new study created in memory with name: no-name-779cef0b-e214-4e85-882e-5d4f63cbf3fc


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5096899224806202
Fold 3 C-index: 0.6787234042553192
Fold 4 C-index: 0.5665399239543726
Fold 5 C-index: 0.5987124463519313
[I 2024-04-13 23:33:37,576] Trial 0 finished with value: 0.5707331394084487 and parameters: {'subsample': 0.7268222670380755, 'dropout_rate': 0.3575254014553415, 'n_estimators': 114, 'learning_rate': 0.05558016213920623}. Best is trial 0 with value: 0.5707331394084487.
Fold 1 C-index: 0.6274900398406374
Fold 2 C-index: 0.5096899224806202
Fold 3 C-index: 0.6787234042553192
Fold 4 C-index: 0.7110266159695817
Fold 5 C-index: 0.5987124463519313
[I 2024-04-13 23:33:48,399] Trial 1 finished with value: 0.6251284857796179 and parameters: {'subsample': 0.7475220728070068, 'dropout_rate': 0.4807958141120149, 'n_estimators': 491, 'learning_rate': 0.06879814411990147}. Best is trial 1 with value: 0.6251284857796179.
Fold 1 C-index: 0.5936254980079682
Fold 2 C-index: 0.5096899224806202
Fold 3 C-index: 0.6893617021276596
Fold 4 C-index: 0.71

Fold 1 C-index: 0.5936254980079682
Fold 2 C-index: 0.5891472868217055
Fold 3 C-index: 0.6893617021276596
Fold 4 C-index: 0.7110266159695817
Fold 5 C-index: 0.5987124463519313
[I 2024-04-13 23:35:07,916] Trial 19 finished with value: 0.6363747098557692 and parameters: {'subsample': 0.2686289669767903, 'dropout_rate': 0.3156718779160812, 'n_estimators': 285, 'learning_rate': 0.018369564224062086}. Best is trial 8 with value: 0.6491050650308138.
Fold 1 C-index: 0.6274900398406374
Fold 2 C-index: 0.5891472868217055
Fold 3 C-index: 0.6893617021276596
Fold 4 C-index: 0.7110266159695817
Fold 5 C-index: 0.5987124463519313
[I 2024-04-13 23:35:10,363] Trial 20 finished with value: 0.6431476182223032 and parameters: {'subsample': 0.28937249949982, 'dropout_rate': 0.7859448591953863, 'n_estimators': 233, 'learning_rate': 0.03672900617406294}. Best is trial 8 with value: 0.6491050650308138.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5096899224806202
Fold 3 C-index: 0.7191489361702128
Fold 4 C-index: 0.7

Fold 1 C-index: 0.6274900398406374
Fold 2 C-index: 0.5096899224806202
Fold 3 C-index: 0.7191489361702128
Fold 4 C-index: 0.7110266159695817
Fold 5 C-index: 0.5987124463519313
[I 2024-04-13 23:35:30,725] Trial 38 finished with value: 0.6332135921625967 and parameters: {'subsample': 0.5612204028878727, 'dropout_rate': 0.18449641578783965, 'n_estimators': 33, 'learning_rate': 0.02899926217525038}. Best is trial 8 with value: 0.6491050650308138.
Fold 1 C-index: 0.6274900398406374
Fold 2 C-index: 0.5891472868217055
Fold 3 C-index: 0.6893617021276596
Fold 4 C-index: 0.7110266159695817
Fold 5 C-index: 0.5987124463519313
[I 2024-04-13 23:35:35,429] Trial 39 finished with value: 0.6431476182223032 and parameters: {'subsample': 0.48668081608198116, 'dropout_rate': 0.40937505489688286, 'n_estimators': 309, 'learning_rate': 0.022700017751066785}. Best is trial 8 with value: 0.6491050650308138.
Fold 1 C-index: 0.6274900398406374
Fold 2 C-index: 0.5891472868217055
Fold 3 C-index: 0.7191489361702128


...

Fold 3 IBS: 0.20009290904253257
Fold 4 IBS: 0.28092759525531996
Fold 5 IBS: 0.21555828062484997
[I 2024-04-13 23:45:25,523] Trial 19 finished with value: 0.2545272898944664 and parameters: {'subsample': 0.3892284838807411, 'dropout_rate': 0.7998935816367629, 'n_estimators': 131, 'learning_rate': 0.04079002339855972}. Best is trial 7 with value: 0.23102462796642137.
Fold 1 IBS: 0.24742760026708696
Fold 2 IBS: 0.2311742087522739
Fold 3 IBS: 0.22734687420564928
Fold 4 IBS: 0.24147435145154503
Fold 5 IBS: 0.22814574868105966
[I 2024-04-13 23:45:26,530] Trial 20 finished with value: 0.23511375667152298 and parameters: {'subsample': 0.6102628884762309, 'dropout_rate': 0.5558690358940764, 'n_estimators': 48, 'learning_rate': 0.0034226676404933015}. Best is trial 7 with value: 0.23102462796642137.
Fold 1 IBS: 0.27050739789833333
Fold 2 IBS: 0.24284075769568764
Fold 3 IBS: 0.19210934491496381
Fold 4 IBS: 0.259467083338547
Fold 5 IBS: 0.20779310091666192
[I 2024-04-13 23:45:27,430] Trial 21 fini

Fold 2 IBS: 0.22880211437250084
Fold 3 IBS: 0.21667863719519237
Fold 4 IBS: 0.24569786120287057
Fold 5 IBS: 0.21931283432048648
[I 2024-04-13 23:47:00,798] Trial 38 finished with value: 0.23220518386473396 and parameters: {'subsample': 0.8179022915704841, 'dropout_rate': 0.20055274347191668, 'n_estimators': 291, 'learning_rate': 0.0051143659180973786}. Best is trial 35 with value: 0.2283205395481725.
Fold 1 IBS: 0.26500588084271626
Fold 2 IBS: 0.2496186566002033
Fold 3 IBS: 0.19950871157267466
Fold 4 IBS: 0.2614732856346855
Fold 5 IBS: 0.20949395998492829
[I 2024-04-13 23:47:07,715] Trial 39 finished with value: 0.23702009892704162 and parameters: {'subsample': 0.2526181938471094, 'dropout_rate': 0.3392786150742839, 'n_estimators': 238, 'learning_rate': 0.01715036845928685}. Best is trial 35 with value: 0.2283205395481725.
Fold 1 IBS: 0.4382567475612011
Fold 2 IBS: 0.37999055787836866
Fold 3 IBS: 0.22265235718715823
Fold 4 IBS: 0.3930613813755423
Fold 5 IBS: 0.33609063559283936
[I 2024

Fold 1 IBS: 0.34134749646168555
Fold 2 IBS: 0.28827179155308336
Fold 3 IBS: 0.19590174214457132
Fold 4 IBS: 0.28445727552826344
Fold 5 IBS: 0.22199440951018173
[I 2024-04-13 23:52:09,634] Trial 57 finished with value: 0.26639454303955706 and parameters: {'subsample': 0.10155250151178148, 'dropout_rate': 0.14123222013912567, 'n_estimators': 414, 'learning_rate': 0.022298480882720552}. Best is trial 42 with value: 0.22691349016324502.
Fold 1 IBS: 0.2863065192864519
Fold 2 IBS: 0.26424025716713934
Fold 3 IBS: 0.19541748908639128
Fold 4 IBS: 0.2691426806122312
Fold 5 IBS: 0.2099161899818892
[I 2024-04-13 23:52:38,765] Trial 58 finished with value: 0.24500462722682056 and parameters: {'subsample': 0.21440673626087092, 'dropout_rate': 0.18587113896354301, 'n_estimators': 463, 'learning_rate': 0.011794289021626232}. Best is trial 42 with value: 0.22691349016324502.
Fold 1 IBS: 0.30737769969344786
Fold 2 IBS: 0.2819266445413722
Fold 3 IBS: 0.20367604257377533
Fold 4 IBS: 0.2939360165255638
Fol

Fold 1 IBS: 0.2516786101504664
Fold 2 IBS: 0.23193557799297
Fold 3 IBS: 0.2020826885904135
Fold 4 IBS: 0.2483972146162672
Fold 5 IBS: 0.21142555143785993
[I 2024-04-13 23:59:48,750] Trial 76 finished with value: 0.2291039285575954 and parameters: {'subsample': 0.19559175095913203, 'dropout_rate': 0.2574951907974749, 'n_estimators': 291, 'learning_rate': 0.009548871684798064}. Best is trial 73 with value: 0.22678321216296818.
Fold 1 IBS: 0.2510429600435854
Fold 2 IBS: 0.23108092412295983
Fold 3 IBS: 0.19844353160525516
Fold 4 IBS: 0.2500563460692217
Fold 5 IBS: 0.2091974778010016
[I 2024-04-14 00:00:20,945] Trial 77 finished with value: 0.2279642479284047 and parameters: {'subsample': 0.12666944775373173, 'dropout_rate': 0.15754313237114592, 'n_estimators': 314, 'learning_rate': 0.011285222373789037}. Best is trial 73 with value: 0.22678321216296818.
Fold 1 IBS: 0.43814619870094235
Fold 2 IBS: 0.3800895010536585
Fold 3 IBS: 0.2274262329574573
Fold 4 IBS: 0.3941020392267966
Fold 5 IBS: 0

Fold 1 IBS: 0.2454644157433154
Fold 2 IBS: 0.22775053358291678
Fold 3 IBS: 0.2049318513399491
Fold 4 IBS: 0.2454285245252791
Fold 5 IBS: 0.2103931376534002
[I 2024-04-14 00:06:05,907] Trial 95 finished with value: 0.2267936925689721 and parameters: {'subsample': 0.10055426267247433, 'dropout_rate': 0.579432905612242, 'n_estimators': 302, 'learning_rate': 0.0104996623949856}. Best is trial 94 with value: 0.22663293718932.
Fold 1 IBS: 0.24407421958765746
Fold 2 IBS: 0.2265178165506223
Fold 3 IBS: 0.21047140976758358
Fold 4 IBS: 0.2402148711696357
Fold 5 IBS: 0.2148373578953299
[I 2024-04-14 00:06:12,260] Trial 96 finished with value: 0.2272231349941658 and parameters: {'subsample': 0.10098443363009074, 'dropout_rate': 0.890178865177957, 'n_estimators': 211, 'learning_rate': 0.010469983994091473}. Best is trial 94 with value: 0.22663293718932.
Fold 1 IBS: 0.2529705416500711
Fold 2 IBS: 0.2321605213728353
Fold 3 IBS: 0.20271441494189463
Fold 4 IBS: 0.24981162098044993
Fold 5 IBS: 0.2114765

In [66]:
train_cindex['ComponentwiseGradientBoosting'] = np.round(study_cindex.best_value, 3)
train_ibs['ComponentwiseGradientBoosting'] = np.round(study_ibs.best_value, 3)

In [67]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.649
train_ibs:  0.227


#### Test

In [68]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [69]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(ComponentwiseGradientBoostingSurvivalAnalysis, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(ComponentwiseGradientBoostingSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

ComponentwiseGradientBoostingSurvivalAnalysis(dropout_rate=0.5443165878852756,
                                              learning_rate=0.03191386107427406,
                                              n_estimators=213,
                                              random_state=123,
                                              subsample=0.4877764869966794)

C-index score: 0.602


ComponentwiseGradientBoostingSurvivalAnalysis(dropout_rate=0.5306326676271426,
                                              learning_rate=0.011681836804668841,
                                              n_estimators=247,
                                              random_state=123,
                                              subsample=0.10169425103438497)

IBS: 0.224


In [70]:
# Saving the values to the dictionary 
test_cindex['ComponentwiseGradientBoosting'] = c_index
test_ibs['ComponentwiseGradientBoosting'] = ibs

## Results

In [71]:
df_train_cindex = pd.DataFrame(train_cindex, index=['C-index']).transpose().sort_values(by='C-index', ascending=False)
df_train_cindex['rank'] = df_train_cindex['C-index'].rank(ascending=False)
df_train_cindex 

,C-index,rank
ComponentwiseGradientBoosting,0.649,1.0
Randomsurvivalforest,0.640,2.5
ExtraSurvivalTrees,0.640,2.5
GradientBoosting,0.637,4.0
CoxPH,0.633,6.0
CoxLasso,0.633,6.0
CoxElastic,0.633,6.0
CoxRidge,0.622,8.0


In [72]:
df_train_ibs = pd.DataFrame(train_ibs, index=['IBS']).transpose().sort_values(by='IBS', ascending=True)
df_train_ibs['rank'] = df_train_ibs['IBS'].rank(ascending=True)
df_train_ibs

,IBS,rank
Randomsurvivalforest,0.219,1.5
ExtraSurvivalTrees,0.219,1.5
CoxLasso,0.223,3.5
CoxElastic,0.223,3.5
CoxPH,0.224,5.0
ComponentwiseGradientBoosting,0.227,6.0
GradientBoosting,0.232,7.0
CoxRidge,0.236,8.0


In [73]:
df_test_cindex = pd.DataFrame(test_cindex, index=['C-index']).transpose().sort_values(by='C-index', ascending=False)
df_test_cindex['rank'] = df_test_cindex['C-index'].rank(ascending=False)
df_test_cindex 

,C-index,rank
ExtraSurvivalTrees,0.654,1.0
CoxRidge,0.653,2.5
Randomsurvivalforest,0.653,2.5
ComponentwiseGradientBoosting,0.602,4.0
GradientBoosting,0.598,5.0
CoxPH,0.550,7.0
CoxLasso,0.550,7.0
CoxElastic,0.550,7.0


In [74]:
df_test_ibs = pd.DataFrame(test_ibs, index=['IBS']).transpose().sort_values(by='IBS', ascending=True)
df_test_ibs['rank'] = df_test_ibs['IBS'].rank(ascending=True)
df_test_ibs

,IBS,rank
Randomsurvivalforest,0.221,1.0
ExtraSurvivalTrees,0.222,2.0
ComponentwiseGradientBoosting,0.224,3.0
GradientBoosting,0.225,4.0
CoxRidge,0.229,5.0
CoxLasso,0.233,6.5
CoxElastic,0.233,6.5
CoxPH,0.235,8.0


In [75]:
# Renaming the column "index" to "model" 
df_train_cindex = df_train_cindex.reset_index().rename(columns={"index": "model"})
df_train_ibs = df_train_ibs.reset_index().rename(columns={"index": "model"})
df_test_cindex = df_test_cindex.reset_index().rename(columns={"index": "model"})
df_test_ibs = df_test_ibs.reset_index().rename(columns={"index": "model"})

# Save the files 
dfs = [df_train_cindex, df_train_ibs, df_test_cindex, df_test_ibs]  # List of your DataFrames
file_path = 'path_to_your_folder/'  # Folder path where you want to save the files

# List of corresponding file names
file_names = ['train_cindex.csv', 'train_ibs.csv', 'test_cindex.csv', 'test_ibs.csv']


dfs = [df_train_cindex, df_train_ibs, df_test_cindex, df_test_ibs]  # List of your DataFrames
file_path = '/Users/minjeongcheon/Desktop/results_thesis/d1/dfs/standard/plsr/'  # Folder path where you want to save the files

# List of corresponding file names
file_names = ['train_cindex.csv', 'train_ibs.csv', 'test_cindex.csv', 'test_ibs.csv']

# Modify the file names to match the desired format
modified_file_names = ['d1_dfs_standard_plsr_' + file_name for file_name in file_names]

# Loop through each DataFrame and save them with corresponding modified file names
for df, modified_file_name in zip(dfs, modified_file_names):
    file_path_name = file_path + modified_file_name  # Construct the full file path
    df.to_csv(file_path_name, index=False)  # Save the DataFrame to CSV file


In [76]:
from datetime import date
today = date.today()
print("Date: ", today)

Date:  2024-04-14
